# LeetCode 988 — Smallest String Starting From Leaf bằng Python

Link đề: https://leetcode.com/problems/smallest-string-starting-from-leaf/ · Độ khó: **Medium** · Chủ đề: String, Backtracking, Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def smallestFromLeaf(self, root: Optional[TreeNode]) -> str:
        ...
```

Ghi chú: đáp án là **một chuỗi duy nhất**. Hai điểm tinh tế của đề: (1) chuỗi đọc **từ lá lên gốc** (nên lá là ký tự đầu tiên); (2) **tiền tố ngắn hơn thì nhỏ hơn** (`"ab" < "aba"`), và một nút chỉ có **một** con **không** phải là lá. Notebook tự dựng `TreeNode`, hàm dựng cây kiểu LeetCode (`build_tree`, duyệt theo tầng, `None` là nút trống) và `to_level`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi nút mang giá trị `0..25` ứng với chữ `'a'..'z'`; tìm chuỗi nhỏ nhất theo thứ tự từ điển trong mọi chuỗi đọc **từ một lá lên tới gốc**.
2. Nắm công cụ: DFS mang theo chuỗi đang dựng (nối **vào đầu**), DFS lặp bằng stack, BFS, con trỏ cha và so sánh trễ.
3. Đi từ **liệt kê mọi lá và đi ngược lên gốc** → **DFS đệ quy mang chuỗi** → **DFS lặp bằng stack** → **BFS** → **so sánh trễ bằng con trỏ cha**.
4. Hiểu mấu chốt: chuỗi của một lá = `(ký tự lá) + … + (ký tự gốc)`; ở mỗi nút ta nối ký tự **vào đầu** chuỗi của đường đi từ gốc xuống; chỉ so sánh tại **lá thật** (nút không có con nào).
5. Tránh lỗi kinh điển: coi nút chỉ có một con là lá, dựng chuỗi theo chiều gốc → lá (ngược), tham lam chọn con nhỏ hơn tại mỗi nút (sai), "lấy chuỗi nhỏ nhất của cây con rồi nối thêm ký tự cha" (sai vì **tiền tố**), so sánh số (`'10'` vs `'2'`), và đệ quy quá sâu (cây lệch tới `8500` nút).

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân, mỗi nút có giá trị trong `[0, 25]` đại diện cho chữ cái `'a'` đến `'z'`.

Trả về chuỗi **nhỏ nhất theo thứ tự từ điển** bắt đầu tại một **lá** của cây và kết thúc tại **gốc**.

Nhắc lại: tiền tố ngắn hơn của một chuỗi thì nhỏ hơn chuỗi đó (`"ab"` nhỏ hơn `"aba"`). Lá là nút không có con.

**Ví dụ 1:** `root = [0,1,2,3,4,3,4]` → `"dba"`.

**Ví dụ 2:** `root = [25,1,3,1,3,0,2]` → `"adz"`.

**Ví dụ 3:** `root = [2,2,1,null,1,0,null,0]` → `"abc"`.

**Ràng buộc:** số nút trong `[1, 8500]`; `0 <= Node.val <= 25`.

## 3. Kiến thức nền

### 3.1 `TreeNode`, dựng cây kiểu LeetCode, `to_level`

LeetCode cung cấp `TreeNode`; ta tự định nghĩa. Danh sách kiểu LeetCode được đọc theo tầng bằng hàng đợi: lấy từng nút đã dựng, gán hai phần tử kế tiếp làm con trái và phải (`None` = không có); **nút trống không có con trong danh sách**.

In [1]:
import itertools
import random
import sys
import time
from collections import deque
from typing import List, Optional

sys.setrecursionlimit(30000)           # cây lệch 8500 nút cần đệ quy sâu (giới hạn mặc định 1000)


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_level(root):
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def tup(root):
    """Dạng bộ lồng nhau (val, trái, phải) — chỉ dùng cho cây nông."""
    return None if root is None else (root.val, tup(root.left), tup(root.right))


def from_tuple(t):
    return None if t is None else TreeNode(t[0], from_tuple(t[1]), from_tuple(t[2]))


def size(root):
    n, stack = 0, [root]
    while stack:
        x = stack.pop()
        if x:
            n += 1
            stack.extend([x.left, x.right])
    return n


r3 = build_tree([2, 2, 1, None, 1, 0, None, 0])
print(tup(r3), to_level(r3))
assert to_level(build_tree([0, 1, 2, 3, 4, 3, 4])) == [0, 1, 2, 3, 4, 3, 4]

(2, (2, None, (1, (0, None, None), None)), (1, (0, None, None), None)) [2, 2, 1, None, 1, 0, None, 0]


### 3.2 Giá trị → chữ cái, so sánh chuỗi trong Python

`chr(ord('a') + v)` đổi `0..25` thành `'a'..'z'`. So sánh chuỗi trong Python đã theo thứ tự từ điển **và** coi tiền tố ngắn là nhỏ hơn: `"ab" < "aba"` là `True`.

In [2]:
def ch(v):
    return chr(ord("a") + v)


print(ch(0), ch(25), "ab" < "aba", "az" > "abz", "abc" < "abcc", min(["dba", "eba", "dca", "eca"]))

a z True True True dba


### 3.3 Lá là gì

Lá = nút **không có con nào** (`left is None and right is None`). Nút có **một** con **không** phải lá: đường đi từ nút đó xuống chưa tới lá. Đây là bẫy hay gặp: với cây chỉ có đường đi `gốc → x`, chuỗi hợp lệ chỉ bắt đầu từ `x`, không phải từ gốc.

In [3]:
chain = build_tree([2, 1, None, 0])         # 2 -> 1 -> 0 (chuỗi trái)
print(tup(chain), "-> lá duy nhất:", chain.left.left.val)

(2, (1, (0, None, None), None), None) -> lá duy nhất: 0


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `root = [0,1,2,3,4,3,4]` là cây đầy đủ 3 tầng: gốc `a`; con `b`, `c`; lá `d`, `e`, `d`, `e`. Các chuỗi từ lá lên gốc: `dba`, `eba`, `dca`, `eca` ⇒ nhỏ nhất `"dba"`.

Ví dụ 3 thể hiện **luật tiền tố**: hai lá cho `"abc"` và `"abcc"`; `"abc"` là tiền tố của `"abcc"` nên **nhỏ hơn** ⇒ đáp án `"abc"`.

Ý tưởng: DFS từ gốc mang theo chuỗi `s` của đường đi từ gốc tới nút hiện tại **đọc ngược** (nút gần gốc ở cuối). Mỗi khi xuống một nút: `s = ch(node.val) + s`. Tại lá, `s` chính là chuỗi lá → gốc; lấy `min`.

In [4]:
def all_leaf_strings(root):
    """Mọi chuỗi lá -> gốc (dùng parent map, không đệ quy)."""
    parent = {id(root): None}
    order, queue = [], deque([root])
    while queue:
        x = queue.popleft()
        order.append(x)
        for c in (x.left, x.right):
            if c is not None:
                parent[id(c)] = x
                queue.append(c)
    out = []
    for x in order:
        if x.left is None and x.right is None:
            s, y = [], x
            while y is not None:
                s.append(ch(y.val))
                y = parent[id(y)]
            out.append("".join(s))
    return out


for l in ([0, 1, 2, 3, 4, 3, 4], [25, 1, 3, 1, 3, 0, 2], [2, 2, 1, None, 1, 0, None, 0]):
    ss = all_leaf_strings(build_tree(l))
    print(l, "->", ss, "-> min:", min(ss))

[0, 1, 2, 3, 4, 3, 4] -> ['dba', 'eba', 'dca', 'eca'] -> min: dba
[25, 1, 3, 1, 3, 0, 2] -> ['bbz', 'dbz', 'adz', 'cdz'] -> min: adz
[2, 2, 1, None, 1, 0, None, 0] -> ['abc', 'abcc'] -> min: abc


## 5. Brute force — liệt kê mọi lá và đi ngược lên gốc

**Ý tưởng:** lập bản đồ cha bằng BFS, xác định mọi lá, với mỗi lá đi ngược lên gốc ghép chuỗi, lấy `min`. Không đệ quy, độc lập với các cách sau — nên dùng làm **oracle**.

In [5]:
def smallest_brute(root):
    return min(all_leaf_strings(root))


for l, want in [([0, 1, 2, 3, 4, 3, 4], "dba"), ([25, 1, 3, 1, 3, 0, 2], "adz"), ([2, 2, 1, None, 1, 0, None, 0], "abc")]:
    assert smallest_brute(build_tree(l)) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [6]:
# Code nộp LeetCode cho Brute force (BFS + con trỏ cha)
class SolutionBrute:
    def smallestFromLeaf(self, root: Optional[TreeNode]) -> str:
        parent = {root: None}
        order, queue = [], [root]
        for x in queue:
            order.append(x)
            for c in (x.left, x.right):
                if c:
                    parent[c] = x
                    queue.append(c)
        best = None
        for x in order:
            if not x.left and not x.right:
                s, y = [], x
                while y:
                    s.append(chr(97 + y.val))
                    y = parent[y]
                cand = "".join(s)
                if best is None or cand < best:
                    best = cand
        return best

**Độ phức tạp:** `O(L · h)` thời gian và bộ nhớ ghép chuỗi với `L` lá, `h` chiều cao (≤ `O(n²)` trường hợp xấu); `O(n)` cho bản đồ cha.

**Nút thắt:** mỗi lá đi ngược lại từ đầu đường đi dù các nút gần gốc dùng chung. Có thể dựng chuỗi **cùng lúc xuống** bằng DFS.

## 6. Bước đột phá

DFS từ gốc mang chuỗi `s` (nối ký tự nút **vào đầu**). Khi tới lá (`not node.left and not node.right`), so sánh `s` với kết quả tốt nhất. Mỗi nút chạm một lần.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** DFS đệ quy, **Cách 3** DFS lặp bằng stack, **Cách 4** BFS, **Cách 5** so sánh trễ bằng con trỏ cha.

### Cách 2 — DFS đệ quy mang chuỗi

**Giải thích từng đoạn:**

1. `dfs(node, s)`: `node` không rỗng; `s` là chuỗi đường đi từ **cha** của `node` ngược lên gốc.
2. `s = ch(node.val) + s`: nối ký tự nút vào đầu.
3. Nếu `node` là lá: `best = min(best, s)`.
4. Ngược lại đệ quy hai con **không rỗng**.

In [7]:
def smallest_dfs(root):
    best = [None]

    def dfs(node, s):
        s = ch(node.val) + s                          # nút gần gốc hơn nằm cuối chuỗi
        if node.left is None and node.right is None:  # lá thật: không có con nào
            if best[0] is None or s < best[0]:
                best[0] = s
            return
        if node.left:
            dfs(node.left, s)
        if node.right:
            dfs(node.right, s)

    dfs(root, "")
    return best[0]


for l, want in [([0, 1, 2, 3, 4, 3, 4], "dba"), ([25, 1, 3, 1, 3, 0, 2], "adz"), ([2, 2, 1, None, 1, 0, None, 0], "abc")]:
    assert smallest_dfs(build_tree(l)) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionDFS:
    def smallestFromLeaf(self, root: Optional[TreeNode]) -> str:
        best = None

        def dfs(node, s):
            nonlocal best
            s = chr(97 + node.val) + s
            if not node.left and not node.right:
                if best is None or s < best:
                    best = s
                return
            if node.left:
                dfs(node.left, s)
            if node.right:
                dfs(node.right, s)

        dfs(root, "")
        return best

**Độ phức tạp:** `O(n · h)` thời gian (mỗi nút nối một chuỗi dài tới `h`), `O(h)` bộ nhớ đệ quy cộng chuỗi hiện tại; `h ≤ 8500` nên **cẩn thận giới hạn đệ quy** của Python (mặc định `1000`).

### Cách 3 — DFS lặp bằng stack

**Ý tưởng:** stack chứa cặp `(node, s)` với `s` là chuỗi **đã gồm** `node`. Lấy ra, nếu lá thì cập nhật đáp án, ngược lại đẩy hai con với chuỗi mới. Không đệ quy, an toàn với cây lệch.

In [9]:
def smallest_iter(root):
    best = None
    stack = [(root, ch(root.val))]
    while stack:
        node, s = stack.pop()
        if node.left is None and node.right is None:
            if best is None or s < best:
                best = s
            continue
        for c in (node.left, node.right):
            if c is not None:
                stack.append((c, ch(c.val) + s))
    return best


for l, want in [([0, 1, 2, 3, 4, 3, 4], "dba"), ([25, 1, 3, 1, 3, 0, 2], "adz"), ([2, 2, 1, None, 1, 0, None, 0], "abc")]:
    assert smallest_iter(build_tree(l)) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [10]:
# Code nộp LeetCode cho Cách 3 (an toàn với cây lệch 8500 nút)
class SolutionIter:
    def smallestFromLeaf(self, root: Optional[TreeNode]) -> str:
        best = None
        stack = [(root, chr(97 + root.val))]
        while stack:
            node, s = stack.pop()
            if not node.left and not node.right:
                if best is None or s < best:
                    best = s
                continue
            for c in (node.left, node.right):
                if c:
                    stack.append((c, chr(97 + c.val) + s))
        return best

**Độ phức tạp:** `O(n · h)` thời gian; `O(n · h)` bộ nhớ trường hợp xấu (nhiều chuỗi trong stack), thực tế nhỏ.

### Cách 4 — BFS

**Ý tưởng:** như Cách 3 nhưng dùng hàng đợi theo tầng; thứ tự duyệt không ảnh hưởng đáp án vì ta lấy `min` trên mọi lá.

In [11]:
def smallest_bfs(root):
    best = None
    queue = deque([(root, ch(root.val))])
    while queue:
        node, s = queue.popleft()
        if node.left is None and node.right is None:
            if best is None or s < best:
                best = s
        for c in (node.left, node.right):
            if c is not None:
                queue.append((c, ch(c.val) + s))
    return best


for l, want in [([0, 1, 2, 3, 4, 3, 4], "dba"), ([25, 1, 3, 1, 3, 0, 2], "adz"), ([2, 2, 1, None, 1, 0, None, 0], "abc")]:
    assert smallest_bfs(build_tree(l)) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [12]:
# Code nộp LeetCode cho Cách 4
from collections import deque


class SolutionBFS:
    def smallestFromLeaf(self, root: Optional[TreeNode]) -> str:
        best = None
        queue = deque([(root, chr(97 + root.val))])
        while queue:
            node, s = queue.popleft()
            if not node.left and not node.right:
                if best is None or s < best:
                    best = s
            for c in (node.left, node.right):
                if c:
                    queue.append((c, chr(97 + c.val) + s))
        return best

**Độ phức tạp:** `O(n · h)` thời gian; bộ nhớ lớn hơn DFS vì hàng đợi chứa nhiều chuỗi cùng tầng (`O(n · h)` xấu nhất).

### Cách 5 — So sánh trễ bằng con trỏ cha

**Ý tưởng:** không dựng chuỗi. Giữ `best_leaf`; với mỗi lá mới, **đi lên đồng thời** từ hai lá và so sánh từng ký tự: dừng ở ký tự khác đầu tiên; nếu một đường đi hết trước (đã tới gốc) thì đường ngắn hơn là tiền tố ⇒ nhỏ hơn. Cuối cùng dựng chuỗi duy nhất của lá tốt nhất.

In [13]:
def smallest_lazy(root):
    parent = {id(root): None}
    leaves, queue = [], deque([root])
    while queue:
        x = queue.popleft()
        if x.left is None and x.right is None:
            leaves.append(x)
        for c in (x.left, x.right):
            if c is not None:
                parent[id(c)] = x
                queue.append(c)

    def less(a, b):
        """Chuỗi lá a -> gốc có nhỏ hơn chuỗi lá b -> gốc không? So sánh trễ từng ký tự."""
        while a is not None and b is not None:
            if a.val != b.val:
                return a.val < b.val
            a, b = parent[id(a)], parent[id(b)]
        return a is None and b is not None            # a hết trước => a là tiền tố thật sự của b => nhỏ hơn

    best = leaves[0]
    for lf in leaves[1:]:
        if less(lf, best):
            best = lf
    s, y = [], best
    while y is not None:
        s.append(ch(y.val))
        y = parent[id(y)]
    return "".join(s)


for l, want in [([0, 1, 2, 3, 4, 3, 4], "dba"), ([25, 1, 3, 1, 3, 0, 2], "adz"), ([2, 2, 1, None, 1, 0, None, 0], "abc")]:
    assert smallest_lazy(build_tree(l)) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [14]:
# Code nộp LeetCode cho Cách 5
class SolutionLazy:
    def smallestFromLeaf(self, root: Optional[TreeNode]) -> str:
        parent = {root: None}
        leaves, queue = [], [root]
        for x in queue:
            if not x.left and not x.right:
                leaves.append(x)
            for c in (x.left, x.right):
                if c:
                    parent[c] = x
                    queue.append(c)

        def less(a, b):
            while a and b:
                if a.val != b.val:
                    return a.val < b.val
                a, b = parent[a], parent[b]
            return a is None and b is not None

        best = leaves[0]
        for lf in leaves[1:]:
            if less(lf, best):
                best = lf
        s = []
        while best:
            s.append(chr(97 + best.val))
            best = parent[best]
        return "".join(s)

**Độ phức tạp:** `O(L · h)` thời gian trường hợp xấu (mỗi lần so sánh đi tới `h` bước, nhưng thường dừng rất sớm); `O(n)` bộ nhớ (bản đồ cha).

## 8. TRACE từng bước

**Trace Cách 2** với ví dụ 3 `root = [2,2,1,null,1,0,null,0]`. Cây: gốc `c`(2); con trái `c`(2) có con phải `b`(1) có con trái `a`(0); con phải `b`(1) có con trái `a`(0).

| đường đi từ gốc | `s` tại lá | lá? |
|---|---|---|
| `c → c → b → a` | `"abcc"` | lá (nút `a` dưới nhánh trái) |
| `c → b → a` | `"abc"` | lá (nút `a` dưới nhánh phải) |

Hai chuỗi `"abcc"` và `"abc"`: `"abc"` là tiền tố ⇒ nhỏ hơn ⇒ đáp án `"abc"`. Hàm trace in từng nút:

In [15]:
def trace_dfs(root):
    best = [None]

    def dfs(node, s, depth):
        s = ch(node.val) + s
        pad = "  " * depth
        if node.left is None and node.right is None:
            better = best[0] is None or s < best[0]
            print(f"{pad}lá {ch(node.val)}: chuỗi {s!r}  ({'cập nhật' if better else 'giữ'} best = {s if better else best[0]!r})")
            if better:
                best[0] = s
            return
        print(f"{pad}nút {ch(node.val)}: s = {s!r}")
        for c in (node.left, node.right):
            if c:
                dfs(c, s, depth + 1)

    dfs(root, "", 0)
    print("=>", best[0])
    return best[0]


assert trace_dfs(build_tree([2, 2, 1, None, 1, 0, None, 0])) == "abc"

nút c: s = 'c'
  nút c: s = 'cc'
    nút b: s = 'bcc'
      lá a: chuỗi 'abcc'  (cập nhật best = 'abcc')
  nút b: s = 'bc'
    lá a: chuỗi 'abc'  (cập nhật best = 'abc')
=> abc


**Trace trường hợp "một con không phải lá":** `root = [0, 1]` (gốc `a` với một con trái `b`): lá duy nhất là `b` nên đáp án `"ba"` (đọc từ lá `b` lên gốc `a`); **không** phải `"a"` (gốc không phải lá vì còn một con).

In [16]:
assert smallest_dfs(build_tree([0, 1])) == "ba" and smallest_dfs(build_tree([5])) == "f"
print(smallest_dfs(build_tree([0, 1])), smallest_dfs(build_tree([5])))

ba f


## 9. Code nộp LeetCode

Chọn **Cách 3** (DFS lặp bằng stack): tránh lỗi đệ quy quá sâu với cây lệch `8500` nút, ngắn gọn. Giữ nguyên `class Solution`, `smallestFromLeaf(self, root)`:

In [17]:
class Solution:
    def smallestFromLeaf(self, root: Optional[TreeNode]) -> str:
        best = None
        stack = [(root, chr(97 + root.val))]          # (nút, chuỗi từ nút ngược lên gốc)
        while stack:
            node, s = stack.pop()
            if not node.left and not node.right:       # lá thật: không có con nào
                if best is None or s < best:           # so sánh từ điển; tiền tố ngắn hơn nhỏ hơn
                    best = s
                continue
            for child in (node.left, node.right):
                if child:
                    stack.append((child, chr(97 + child.val) + s))   # nối ký tự con vào ĐẦU
        return best


tests3 = [([0, 1, 2, 3, 4, 3, 4], "dba"), ([25, 1, 3, 1, 3, 0, 2], "adz"), ([2, 2, 1, None, 1, 0, None, 0], "abc")]
print([Solution().smallestFromLeaf(build_tree(l)) for l, _ in tests3])
assert all(Solution().smallestFromLeaf(build_tree(l)) == w for l, w in tests3)

['dba', 'adz', 'abc']


## 10. Vì sao đúng

**(a) Đề tương đương "min trên mọi lá".** Mỗi chuỗi hợp lệ ứng với đúng một lá `x`: `S(x) = ch(x) ch(parent(x)) … ch(root)`. Đáp án là `min_x S(x)` theo thứ tự từ điển của Python (tiền tố ngắn hơn nhỏ hơn) — khớp phát biểu đề.

**(b) DFS/BFS/stack dựng đúng `S(x)`.** Quy nạp theo độ sâu: tại gốc `s = ch(root)`; khi xuống con `c` của `u` có `s_u = ch(u) … ch(root)`, ta đặt `s_c = ch(c) + s_u = ch(c) ch(u) … ch(root)`. Tại lá `s = S(leaf)`. Mọi lá được thăm đúng một lần (mỗi nút được thăm một lần vì cây không có chu trình) ⇒ `min` đúng.

**(c) Chỉ so sánh tại lá thật.** Nút có một con không phải lá: chuỗi kết thúc tại nút đó **không** bắt đầu từ lá nên không hợp lệ. Điều kiện `not node.left and not node.right` đảm bảo điều này. Cây có một nút: gốc cũng là lá ⇒ chuỗi `ch(root)`.

**(d) Vì sao không thể tham lam hay "chuỗi nhỏ nhất của cây con + ký tự cha".** Với `s₁ = "a"` và `s₂ = "ab"` ở hai cây con, sau khi nối ký tự `z` của cha: `"az"` và `"abz"`; `s₁ < s₂` nhưng `"az" > "abz"`. Nên chuỗi nhỏ nhất của cây con **không** chắc dẫn tới chuỗi nhỏ nhất của cây lớn — phải xét theo đủ **mọi chuỗi** từ lá tới gốc (hoặc hoãn so sánh đến khi biết đoạn nối). Tương tự, đi xuống con có giá trị nhỏ hơn (tham lam trên đường đi từ gốc) sai vì chuỗi bắt đầu từ lá chứ không từ gốc.

**(e) Cách 5.** `less(a, b)` so sánh `S(a)` và `S(b)` từng ký tự bằng cách đi lên cùng lúc: ký tự khác đầu tiên quyết định; nếu `a` hết (đã vượt gốc) mà `b` còn thì `S(a)` là tiền tố thật sự của `S(b)` ⇒ nhỏ hơn; nếu cả hai hết cùng lúc thì bằng nhau (`less` trả `False`). Giữ lá tốt nhất ⇒ `min`. Chuỗi của lá tốt nhất được dựng một lần ở cuối.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp oracle (BFS + con trỏ cha, độc lập đệ quy) và một oracle thứ hai **đệ quy trên bộ lồng nhau** trên **mọi** cây ≤ 5 nút với nhãn `{a, b, z}` (tức `0, 1, 25`: cố ý có `"az"` vs `"abz"`), tổng `11 496` cây; khớp nhau trên cây ngẫu nhiên tới `8500` nút, kể cả cây lệch (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lá → gốc (cha) | `O(L·h)` | `O(n + h)` | oracle |
| 2. DFS đệ quy | `O(n·h)` | `O(h)` | đệ quy sâu `≤ 8500` |
| 3. DFS lặp | `O(n·h)` | `O(n·h)` xấu | **đáp án chính**, an toàn |
| 4. BFS | `O(n·h)` | `O(n·h)` xấu | theo tầng |
| 5. So sánh trễ | `O(L·h)` xấu | `O(n)` | không dựng chuỗi |

(`L` số lá, `h` chiều cao.) Đo thực tế: cây ngẫu nhiên `8500` nút, cây đầy đủ, cây lệch (chuỗi) `8500` nút:

In [18]:
def random_tree(rng, n, shape="random", alphabet=26):
    """Cây n nút (dựng LẶP, không đệ quy). shape: random | chain | left_comb | complete | caterpillar."""
    if n == 0:
        return None
    vals = [rng.randrange(alphabet) for _ in range(n)]
    nodes = [TreeNode(vals[0])]
    free = [(0, 0), (0, 1)]                              # (chỉ số nút, 0 = trái / 1 = phải) còn trống
    for i in range(1, n):
        node = TreeNode(vals[i])
        nodes.append(node)
        if shape == "random":
            k = rng.randrange(len(free))
            free[k], free[-1] = free[-1], free[k]
            p, side = free.pop()
        elif shape == "chain":                           # mỗi nút một con, hướng ngẫu nhiên
            p, side = (i - 1, rng.randrange(2))
        elif shape == "left_comb":                       # sống lưng trái, mỗi nút trên sống lưng có thêm lá phải
            if i % 2 == 1:
                p, side = (i - 1 if i == 1 else i - 2, 0)
            else:
                p, side = (i - 2 if i > 2 else 0, 1)
        elif shape == "complete":
            p, side = ((i - 1) // 2, (i - 1) % 2)
        else:
            raise ValueError(shape)
        if side == 0:
            nodes[p].left = node
        else:
            nodes[p].right = node
        if shape == "random":
            free.extend([(i, 0), (i, 1)])
    return nodes[0]


def depth(root):
    best, stack = 0, [(root, 1)]
    while stack:
        x, d = stack.pop()
        if x:
            best = max(best, d)
            stack.extend([(x.left, d + 1), (x.right, d + 1)])
    return best


def bench(fn, root, repeat=3):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(root)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(988)
shapes = {name: random_tree(rng, 8500, name) for name in ("random", "complete", "chain")}
for name, tr in shapes.items():
    print(f"{name:9s}: n = {size(tr)}, độ sâu = {depth(tr)}")
    for fname, fn in [("dfs", smallest_dfs), ("iter", smallest_iter), ("bfs", smallest_bfs), ("lazy", smallest_lazy)]:
        print(f"    {fname:5s}: {bench(fn, tr) * 1e3:8.2f} ms")

random   : n = 8500, độ sâu = 30
    dfs  :     1.81 ms
    iter :     1.68 ms


    bfs  :     2.15 ms


    lazy :     3.47 ms
complete : n = 8500, độ sâu = 14


    dfs  :     2.25 ms


    iter :     3.10 ms


    bfs  :     3.72 ms
    lazy :     1.83 ms


chain    : n = 8500, độ sâu = 8500
    dfs  :     4.00 ms


    iter :     2.60 ms


    bfs  :     2.66 ms


    lazy :     2.49 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi cây ≤ 5 nút với nhãn `{0, 1, 25}`, (E) tính chất / ngẫu nhiên (cây tới 8500 nút, seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [19]:
IMPLS = {
    "dfs": smallest_dfs,
    "iter": smallest_iter,
    "bfs": smallest_bfs,
    "lazy": smallest_lazy,
    "brute": smallest_brute,
    "Solution": lambda r: Solution().smallestFromLeaf(r),
    "SolutionDFS": lambda r: SolutionDFS().smallestFromLeaf(r),
    "SolutionIter": lambda r: SolutionIter().smallestFromLeaf(r),
    "SolutionBFS": lambda r: SolutionBFS().smallestFromLeaf(r),
    "SolutionLazy": lambda r: SolutionLazy().smallestFromLeaf(r),
    "SolutionBrute": lambda r: SolutionBrute().smallestFromLeaf(r),
}


def oracle_tuple(t):
    """Oracle thứ hai: đệ quy trên bộ lồng nhau (val, trái, phải) — không dùng con trỏ cha/ngăn xếp."""
    def paths(node):
        if node is None:
            return []
        v, l, r = node
        if l is None and r is None:
            return [ch(v)]
        return [p + ch(v) for p in paths(l) + paths(r)]      # chuỗi lá -> nút: nối ký tự nút vào CUỐI

    return min(paths(t))


def check(l, want=None, names=None):
    root = build_tree(l)
    if want is None:
        want = smallest_brute(root)
        if size(root) <= 200:
            assert want == oracle_tuple(tup(root))
    for name in (names or IMPLS):
        got = IMPLS[name](build_tree(l))
        assert got == want, (name, l[:12], got, want)


for l, w in tests3:
    check(l, w)
print("ví dụ đề: hai oracle và", len(IMPLS), "cài đặt đúng")

ví dụ đề: hai oracle và 11 cài đặt đúng


### B. Edge case

- Một nút: `[0]` → `"a"`, `[25]` → `"z"`.
- Gốc chỉ có một con (không phải lá): `[0, 1]` → `"ba"`; `[1, None, 0]` → `"ab"`.
- Chuỗi (mỗi nút một con) dài.
- Luật tiền tố: `"abc"` nhỏ hơn `"abcc"` (ví dụ 3).
- Cùng ký tự lá, khác phần trên (so sánh sâu).
- Hai lá cho cùng chuỗi.
- Nút giữa có chỉ một con (không được coi là lá).
- Giá trị `25` (`z`) ở gốc và `0` (`a`) ở lá.

In [20]:
edge = [
    [0], [25], [1, 0], [1, None, 0], [2, 1, None, 0], [2, None, 1, None, 0], [25, 0, 0], [0, 25, 25],
    [2, 2, 1, None, 1, 0, None, 0], [25, 1, 3, 1, 3, 0, 2], [0, 1, 2, 3, 4, 3, 4], [1, 0, 0, 1], [3, 1, 2, 0, None, None, 1],
    [0, 1, None, 2, None, 3], [5, 3, None, 3, None, 3, None, 3], [25, 0, None, 1, None, 0], [0, 0, 0, 0, 0, 0, 0],
    [1, 2, 3, 4, None, None, 5, 6],
]
for l in edge:
    check(l)
print(len(edge), "edge case: mọi cài đặt khớp oracle")
print("[3,1,2,0,None,None,1] ->", Solution().smallestFromLeaf(build_tree([3, 1, 2, 0, None, None, 1])))

18 edge case: mọi cài đặt khớp oracle
[3,1,2,0,None,None,1] -> abd


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [21]:
def bug_one_child_is_leaf(root):                      # coi nút thiếu MỘT con là lá
    best = None

    def dfs(node, s):
        nonlocal best
        s = ch(node.val) + s
        if node.left is None or node.right is None:
            if best is None or s < best:
                best = s
            if node.left is None and node.right is None:
                return
        if node.left:
            dfs(node.left, s)
        if node.right:
            dfs(node.right, s)

    dfs(root, "")
    return best


def bug_root_to_leaf(root):                           # dựng chuỗi theo chiều gốc -> lá (nối vào CUỐI)
    best = None

    def dfs(node, s):
        nonlocal best
        s = s + ch(node.val)
        if node.left is None and node.right is None:
            if best is None or s < best:
                best = s
            return
        if node.left:
            dfs(node.left, s)
        if node.right:
            dfs(node.right, s)

    dfs(root, "")
    return best


def bug_greedy_child(root):                           # tại mỗi nút đi xuống con có giá trị nhỏ hơn
    s = ""
    node = root
    path = []
    while node:
        path.append(ch(node.val))
        kids = [c for c in (node.left, node.right) if c]
        node = min(kids, key=lambda c: c.val) if kids else None
    return "".join(reversed(path))


def bug_bottom_up_min(root):                          # chuỗi nhỏ nhất của cây con rồi nối ký tự cha (sai vì tiền tố)
    def go(node):
        kids = [c for c in (node.left, node.right) if c]
        if not kids:
            return ch(node.val)
        return min(go(c) for c in kids) + ch(node.val)

    return go(root)


def bug_numeric_compare(root):                        # so sánh theo chuỗi số '10' < '2' thay vì chữ cái
    best = None

    def dfs(node, s):
        nonlocal best
        s = [node.val] + s
        if node.left is None and node.right is None:
            key = "".join(str(v) for v in s)
            if best is None or key < best[0]:
                best = (key, "".join(ch(v) for v in s))
            return
        if node.left:
            dfs(node.left, s)
        if node.right:
            dfs(node.right, s)

    dfs(root, [])
    return best[1]


def bug_longest_wins(root):                           # chuỗi dài hơn được coi là nhỏ hơn khi một chuỗi là tiền tố của chuỗi kia
    best = None

    def dfs(node, s):
        nonlocal best
        s = ch(node.val) + s
        if node.left is None and node.right is None:
            if best is None or (s < best and not best.startswith(s)) or s.startswith(best):
                best = s
            return
        if node.left:
            dfs(node.left, s)
        if node.right:
            dfs(node.right, s)

    dfs(root, "")
    return best


def bug_max(root):                                    # lấy chuỗi LỚN nhất
    return max(all_leaf_strings(root))


BUGS = [bug_one_child_is_leaf, bug_root_to_leaf, bug_greedy_child, bug_bottom_up_min, bug_numeric_compare, bug_longest_wins, bug_max]
probe_small = [[1, 0], [0, 1, 2, 3, 4, 3, 4], [25, 1, 3, 1, 3, 0, 2], [2, 2, 1, None, 1, 0, None, 0], [3, 1, 2, 0, None, None, 1], [25, 0, 1, None, 1],
               [2, 25, 1, 1, 0], [0, 12, 2, None, None, 1, None]]
for bug in BUGS:
    found = None
    for l in probe_small:
        try:
            got = bug(build_tree(l))
        except Exception as e:
            got = type(e).__name__
        if got != smallest_brute(build_tree(l)):
            found = (l, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:22s} sai trên {found[0]}: ra {found[1]!r}, đúng {smallest_brute(build_tree(found[0]))!r}")

bug_one_child_is_leaf  sai trên [25, 0, 1, None, 1]: ra 'az', đúng 'baz'
bug_root_to_leaf       sai trên [1, 0]: ra 'ba', đúng 'ab'
bug_greedy_child       sai trên [25, 1, 3, 1, 3, 0, 2]: ra 'bbz', đúng 'adz'
bug_bottom_up_min      sai trên [25, 0, 1, None, 1]: ra 'bz', đúng 'baz'
bug_numeric_compare    sai trên [0, 12, 2, None, None, 1, None]: ra 'ma', đúng 'bca'
bug_longest_wins       sai trên [2, 2, 1, None, 1, 0, None, 0]: ra 'abcc', đúng 'abc'
bug_max                sai trên [0, 1, 2, 3, 4, 3, 4]: ra 'eca', đúng 'dba'


### D. Vét cạn mọi cây ≤ 5 nút với nhãn `{0, 1, 25}`

`all_trees(n, alphabet)` sinh mọi cây `n` nút: chọn nhãn gốc, chia `n−1` nút còn lại cho trái/phải theo mọi cách, đệ quy. Số cây = Catalan(n) × 3ⁿ. Tổng với `n = 1..5`: `3 + 18 + 135 + 1134 + 10206 = 11 496`. Hai oracle: `smallest_brute` (BFS + cha) và `oracle_tuple` (đệ quy trên bộ lồng nhau) phải khớp, rồi mọi cài đặt khớp chúng.

In [22]:
def all_trees(n, alphabet):
    if n == 0:
        return [None]
    out = []
    for k in range(n):
        for lt in all_trees(k, alphabet):
            for rt in all_trees(n - 1 - k, alphabet):
                for v in alphabet:
                    out.append((v, lt, rt))
    return out


t0 = time.perf_counter()
total = 0
for n in range(1, 6):
    trees = all_trees(n, (0, 1, 25))
    for t in trees:
        want = oracle_tuple(t)
        root = from_tuple(t)
        assert smallest_brute(root) == want, (t, want)
        for name in IMPLS:
            assert IMPLS[name](root) == want, (name, t, IMPLS[name](root), want)
    total += len(trees)
    print(f"  n = {n}: {len(trees)} cây")
print(total, "cây:", len(IMPLS), "cài đặt khớp hai oracle;", round(time.perf_counter() - t0, 1), "s")
assert total == 3 + 18 + 135 + 1134 + 10206

  n = 1: 3 cây
  n = 2: 18 cây
  n = 3: 135 cây


  n = 4: 1134 cây


  n = 5: 10206 cây
11496 cây: 11 cài đặt khớp hai oracle; 0.4 s


### E. Tính chất / ngẫu nhiên (cây tới 8500 nút)

Seed cố định; bảng chữ cái nhỏ (`2`–`4` chữ) để tạo nhiều hòa và nhiều trường hợp tiền tố; các hình dạng `random`, `chain`, `left_comb`, `complete`. Kiểm tra: (1) các cách khớp nhau; với `n ≤ 2000` khớp brute; (2) kết quả là chuỗi của một lá thật (có một lá cho đúng chuỗi đó) và **không có lá nào cho chuỗi nhỏ hơn** (kiểm tra trực tiếp bằng `all_leaf_strings`); (3) độ dài kết quả `≤` độ sâu cây; (4) kết quả chỉ gồm chữ trong bảng chữ cái đã dùng; (5) đổi chỗ hai con ở **mọi** nút không đổi tập chuỗi lá nên đáp án không đổi.

In [23]:
def mirror(root):
    """Bản sao với hai con đổi chỗ ở mọi nút (lặp)."""
    if root is None:
        return None
    new_root = TreeNode(root.val)
    stack = [(root, new_root)]
    while stack:
        a, b = stack.pop()
        if a.left:
            b.right = TreeNode(a.left.val)
            stack.append((a.left, b.right))
        if a.right:
            b.left = TreeNode(a.right.val)
            stack.append((a.right, b.left))
    return new_root


rng = random.Random(20240988)
t0 = time.perf_counter()
fast = ["dfs", "iter", "bfs", "Solution", "SolutionDFS", "SolutionIter", "SolutionBFS"]
for it in range(120):
    n = rng.choice([1, 2, 3, 5, 10, 50, 200, 1000, 2000, 8500])
    shape = rng.choice(["random", "chain", "left_comb", "complete"])
    alphabet = rng.choice([2, 3, 4, 26])
    tree = random_tree(rng, n, shape, alphabet)
    want = IMPLS["Solution"](tree)
    for name in fast:
        assert IMPLS[name](tree) == want, (name, it, shape, n)                          # (1)
    if n <= 2000:
        leaves = all_leaf_strings(tree)
        assert want == min(leaves) == IMPLS["lazy"](tree) == IMPLS["brute"](tree)       # (1) (2)
        assert want in leaves and all(want <= s for s in leaves)                        # (2)
    assert len(want) <= depth(tree)                                                     # (3)
    assert set(want) <= {ch(v) for v in range(alphabet)}                                # (4)
    assert IMPLS["Solution"](mirror(tree)) == want                                      # (5)
print("120 cây ngẫu nhiên (n tới 8500, 4 hình dạng, bảng chữ cái nhỏ): 5 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

120 cây ngẫu nhiên (n tới 8500, 4 hình dạng, bảng chữ cái nhỏ): 5 tính chất đúng; 0.5 s


In [24]:
# cây nhỏ ngẫu nhiên: so với oracle đệ quy trên bộ lồng nhau cho mọi cài đặt
rng = random.Random(98802)
t0 = time.perf_counter()
for _ in range(1500):
    n = rng.randint(1, 40)
    tree = random_tree(rng, n, rng.choice(["random", "chain", "left_comb", "complete"]), rng.choice([2, 3, 26]))
    want = oracle_tuple(tup(tree))
    for name in IMPLS:
        assert IMPLS[name](tree) == want, (name, to_level(tree), IMPLS[name](tree), want)
print("1500 cây nhỏ ngẫu nhiên: mọi cài đặt khớp oracle đệ quy;", round(time.perf_counter() - t0, 1), "s")

1500 cây nhỏ ngẫu nhiên: mọi cài đặt khớp oracle đệ quy; 0.2 s


### F. Giới hạn

`n = 8500`: cây lệch (chuỗi), cây sống lưng (caterpillar/`left_comb`), cây đầy đủ, ngẫu nhiên; trường hợp tất cả giá trị bằng nhau (nhiều hòa; luật tiền tố quyết định: chuỗi **ngắn nhất** thắng); giá trị `25` toàn bộ.

In [25]:
t0 = time.perf_counter()
rng = random.Random(98803)
n = 8500
for shape in ("chain", "left_comb", "complete", "random"):
    tree = random_tree(rng, n, shape, 26)
    results = {name: IMPLS[name](tree) for name in ("dfs", "iter", "bfs", "Solution")}
    assert len(set(results.values())) == 1, shape
    print(f"  {shape:9s}: độ sâu {depth(tree):5d}, độ dài đáp án {len(results['Solution'])}")
same = random_tree(rng, n, "left_comb", 1)                       # toàn 'a'
res = {name: IMPLS[name](same) for name in ("dfs", "iter", "bfs", "Solution")}
assert len(set(res.values())) == 1
leaves = all_leaf_strings(same)
assert res["Solution"] == "a" * min(len(s) for s in leaves) == min(leaves)       # toàn bằng nhau: chuỗi ngắn nhất thắng
zz = random_tree(rng, 3000, "random", 1)
for x in [zz]:
    stack = [x]
    while stack:
        y = stack.pop()
        if y:
            y.val = 25
            stack.extend([y.left, y.right])
assert set(IMPLS["Solution"](zz)) == {"z"}
print("n = 8500, 4 hình dạng + toàn 'a' + toàn 'z': mọi cách khớp;", round(time.perf_counter() - t0, 1), "s")

  chain    : độ sâu  8500, độ dài đáp án 8500


  left_comb: độ sâu  4251, độ dài đáp án 4250
  complete : độ sâu    14, độ dài đáp án 13
  random   : độ sâu    32, độ dài đáp án 17
n = 8500, 4 hình dạng + toàn 'a' + toàn 'z': mọi cách khớp; 0.1 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: coi nút chỉ có một con là lá

In [26]:
t = build_tree([0, 1])
print("bug:", bug_one_child_is_leaf(t), "| đúng:", Solution().smallestFromLeaf(t))
assert bug_one_child_is_leaf(t) == "a" and Solution().smallestFromLeaf(t) == "ba"

bug: a | đúng: ba


**Vì sao sai:** gốc `a` còn con `b` nên không phải lá; chuỗi hợp lệ duy nhất bắt đầu từ lá `b`: `"ba"` (bản sai trả `"a"` vì coi gốc là lá). **Đúng:** lá ⇔ **cả hai** con đều `None`.

### Lỗi 2: dựng chuỗi theo chiều gốc → lá

In [27]:
t = build_tree([0, 1, 2, 3, 4, 3, 4])
print("bug:", bug_root_to_leaf(t), "| đúng:", Solution().smallestFromLeaf(t))
assert bug_root_to_leaf(t) == "abd" and Solution().smallestFromLeaf(t) == "dba"

bug: abd | đúng: dba


**Vì sao sai:** đề đọc **từ lá lên gốc**; ký tự đầu là ký tự của lá. **Đúng:** nối ký tự nút **vào đầu** chuỗi (`s = ch + s`).

### Lỗi 3: tham lam đi xuống con nhỏ hơn

In [28]:
t = build_tree([1, 0, 0, None, None, None, 0])        # gốc b; con trái a (lá); con phải a có con phải a
print("các chuỗi lá:", all_leaf_strings(t))
print("bug:", bug_greedy_child(t), "| đúng:", Solution().smallestFromLeaf(t))
assert bug_greedy_child(t) == "ab" and Solution().smallestFromLeaf(t) == "aab"

các chuỗi lá: ['ab', 'aab']
bug: ab | đúng: aab


**Vì sao sai:** chuỗi bắt đầu từ **lá**, không từ gốc; chọn con nhỏ hơn ở gần gốc chẳng đảm bảo lá nhỏ (ở đây hai con đều là `a`, nhưng chỉ nhánh phải dẫn tới chuỗi `"aab"` nhỏ hơn `"ab"`). Phải xét mọi lá.

### Lỗi 4: "chuỗi nhỏ nhất của cây con + ký tự cha" (bẫy tiền tố)

In [29]:
t = build_tree([1, 0, 0, None, None, None, 0])        # cùng cây: hai chuỗi lá "ab" và "aab"
print("các chuỗi lá:", all_leaf_strings(t))
print("bug (min cây con + ký tự cha):", bug_bottom_up_min(t), "| đúng:", Solution().smallestFromLeaf(t))
assert bug_bottom_up_min(t) == "ab" and Solution().smallestFromLeaf(t) == "aab"

các chuỗi lá: ['ab', 'aab']
bug (min cây con + ký tự cha): ab | đúng: aab


In [30]:
# tìm phản ví dụ nhỏ nhất của bug_bottom_up_min bằng vét cạn (cây ≤ 4 nút, nhãn {0,1,25})
found = None
for n in range(1, 5):
    for t in all_trees(n, (0, 1, 25)):
        if bug_bottom_up_min(from_tuple(t)) != oracle_tuple(t):
            found = t
            break
    if found:
        break
print("cây:", found, "| bug:", bug_bottom_up_min(from_tuple(found)), "| đúng:", oracle_tuple(found), "| các chuỗi lá:", all_leaf_strings(from_tuple(found)))
assert found is not None

cây: (1, (0, None, None), (0, None, (0, None, None))) | bug: ab | đúng: aab | các chuỗi lá: ['ab', 'aab']


**Vì sao sai:** với hai chuỗi con `s₁ = "a"` và `s₂ = "ab"` (tiền tố), sau khi nối `"z"` của cha: `"az"` và `"abz"`; `s₁ < s₂` nhưng `"az" > "abz"`. Chuỗi con nhỏ nhất không chắc cho chuỗi cuối nhỏ nhất. **Đúng:** so sánh **chuỗi đầy đủ lá → gốc**.

### Lỗi 5: so sánh số thay vì chữ cái

In [31]:
t = build_tree([0, 10, 2])
print("bug:", bug_numeric_compare(t), "| đúng:", Solution().smallestFromLeaf(t), "| các chuỗi lá:", all_leaf_strings(t))
assert bug_numeric_compare(t) != Solution().smallestFromLeaf(t)

bug: ka | đúng: ca | các chuỗi lá: ['ka', 'ca']


**Vì sao sai:** ghép số `'10'` và `'2'` rồi so sánh chuỗi cho `'10' < '2'` trong khi chữ cái `'k'` (10) lớn hơn `'c'` (2). **Đúng:** đổi mỗi giá trị sang **một** chữ cái trước khi so sánh.

### Lỗi 6 (bonus): đệ quy quá sâu với cây lệch

In [32]:
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
deep = random_tree(random.Random(1), 1500, "chain", 3)
try:
    smallest_dfs(deep)
    print("không lỗi (môi trường cho phép đệ quy sâu)")
except RecursionError:
    print("RecursionError với đệ quy sâu 1500; bản lặp vẫn chạy:", len(smallest_iter(deep)))
finally:
    sys.setrecursionlimit(old)
assert smallest_iter(deep) == smallest_brute(deep)

RecursionError với đệ quy sâu 1500; bản lặp vẫn chạy: 1500


**Vì sao sai:** cây có thể sâu tới `8500` nút; đệ quy Python mặc định giới hạn `1000`. **Đúng:** DFS lặp bằng stack (Cách 3) hoặc tăng `sys.setrecursionlimit`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chuỗi nhỏ nhất nối các nút theo đường đi cây, từ lá lên gốc" → DFS/BFS mang chuỗi, nối ký tự **vào đầu**, so sánh tại **lá thật**.

**Mẹo:**

1. Lá = không có **con nào**; nút một con không phải lá.
2. `s = ch + s` để chuỗi tự "đảo" thành lá → gốc.
3. So sánh chuỗi Python đã coi tiền tố ngắn là nhỏ hơn.
4. Không tối ưu cục bộ ("nhỏ nhất của cây con") khi có luật tiền tố.
5. DFS lặp tránh `RecursionError` cho cây lệch.
6. Oracle: liệt kê mọi lá (BFS + con trỏ cha) và `min`.

**Họ bài cùng pattern:** Sum Root to Leaf Numbers (129), Binary Tree Paths (257), Path Sum II (113), Smallest String With Swaps, Lexicographically Smallest Equivalent String (1061), Sum of Root To Leaf Binary Numbers (1022).

## 15. Biến thể

1. **Chuỗi lớn nhất** từ lá lên gốc.
2. **Chuỗi nhỏ nhất đọc từ gốc xuống lá** (đường đi gốc → lá).
3. **Trả về cả đường đi** (danh sách giá trị từ lá lên gốc) thay vì chuỗi.

Cài đặt và đối chiếu vét cạn:

In [33]:
def largest_from_leaf(root):
    best, stack = None, [(root, ch(root.val))]
    while stack:
        node, s = stack.pop()
        if not node.left and not node.right:
            best = s if best is None or s > best else best
            continue
        for c in (node.left, node.right):
            if c:
                stack.append((c, ch(c.val) + s))
    return best


def smallest_root_to_leaf(root):
    best, stack = None, [(root, ch(root.val))]
    while stack:
        node, s = stack.pop()
        if not node.left and not node.right:
            best = s if best is None or s < best else best
            continue
        for c in (node.left, node.right):
            if c:
                stack.append((c, s + ch(c.val)))
    return best


def smallest_path_values(root):
    return [ord(c) - 97 for c in smallest_dfs(root)]


for n in range(1, 5):
    for t in all_trees(n, (0, 1, 25)):
        root = from_tuple(t)
        leaves = all_leaf_strings(root)
        assert largest_from_leaf(root) == max(leaves)
        assert smallest_root_to_leaf(root) == min(s[::-1] for s in leaves)
        path = smallest_path_values(root)
        assert "".join(ch(v) for v in path) == oracle_tuple(t)
print("chuỗi lớn nhất, root->leaf nhỏ nhất, đường đi giá trị: khớp vét cạn")

chuỗi lớn nhất, root->leaf nhỏ nhất, đường đi giá trị: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lá → gốc (cha) | liệt kê lá, đi ngược | `O(L·h)` | `O(n+h)` | oracle |
| 2. DFS đệ quy | mang chuỗi, nối vào đầu | `O(n·h)` | `O(h)` | gọn, cây nông |
| 3. DFS lặp | stack `(nút, chuỗi)` | `O(n·h)` | `O(n·h)` xấu | **đáp án chính** |
| 4. BFS | queue `(nút, chuỗi)` | `O(n·h)` | `O(n·h)` xấu | theo tầng |
| 5. So sánh trễ | con trỏ cha, không dựng chuỗi | `O(L·h)` xấu | `O(n)` | tiết kiệm bộ nhớ chuỗi |

Mọi cách khớp hai oracle trên `11 496` cây nhỏ (nhãn `{a, b, z}`) và khớp nhau tới `8500` nút.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [34]:
print(Solution().smallestFromLeaf(build_tree([0])), Solution().smallestFromLeaf(build_tree([1, 0])), Solution().smallestFromLeaf(build_tree([2, 1, None, 0])),
      Solution().smallestFromLeaf(build_tree([25, 0, 0])))

a ab abc az


### Bài 2 (tính tay)

`root = [3, 1, 2, 0, null, null, 1]` (gốc `d`; con trái `b` có con trái `a`; con phải `c` có con phải `b`). (a) Các lá là gì? (b) Chuỗi từ mỗi lá lên gốc? (c) Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DFS đệ quy:

In [35]:
def dfs_todo(root):
    best = [None]

    def dfs(node, s):
        s = ""                  # TODO 1: nối ký tự của node vào ĐẦU s (dùng chr(97 + node.val))
        if False:               # TODO 2: node là lá thật (cả hai con đều None)
            pass                # TODO 3: cập nhật best[0] nếu s nhỏ hơn
        # TODO 4: đệ quy sang con trái và phải nếu tồn tại

    dfs(root, "")
    return best[0]


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(dfs_todo(build_tree(l)) == w for l, w in tests3)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **BFS** (hàng đợi `(nút, chuỗi)`):

In [36]:
class MySolution:
    def smallestFromLeaf(self, root):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().smallestFromLeaf(build_tree(l)) == w for l, w in tests3)

### Bài 5 (mini challenge): chuỗi nhỏ nhất từ gốc xuống lá

Viết `smallest_down(root)` trả về chuỗi nhỏ nhất đọc **từ gốc xuống lá** (đường đi gốc → lá, lá là nút không có con). Gợi ý: nối ký tự vào **cuối**. Kiểm bằng vét cạn ở mục 15.

In [37]:
def smallest_down(root):
    pass            # TODO


# Khi làm xong:
# assert smallest_down(build_tree([0, 1, 2, 3, 4, 3, 4])) == "abd" and smallest_down(build_tree([1, 0])) == "ba"

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: một nút là lá; chuỗi gồm đúng ký tự đó.
- Mức 2: `[1, 0]` có một lá `a`, đọc lên gốc `b`.
- Mức 3: `a ab abc az`.

**Bài 2**

- Mức 1: lá là nút không có con: giá trị `0` (con của `1`) và `1` (con của `2`).
- Mức 2: lá `0` → `a b d`; lá `1` → `b c d`.
- Mức 3: `"abd"` và `"bcd"` ⇒ đáp án `"abd"`.

**Bài 3**

- Mức 1: `s = chr(97 + node.val) + s`.
- Mức 2: `if not node.left and not node.right:`.
- Mức 3: `if best[0] is None or s < best[0]: best[0] = s`; rồi `if node.left: dfs(node.left, s)` và tương tự bên phải.

**Bài 4**

- Mức 1: `queue = deque([(root, chr(97 + root.val))])`.
- Mức 2: lá thì so sánh; không phải lá thì đẩy con với chuỗi `chr(97 + c.val) + s`.
- Mức 3: trả `best`.

**Bài 5**

- Mức 1: giống bài chính nhưng nối ký tự vào **cuối** (`s + ch`).
- Mức 2: lá vẫn là nút không có con nào.
- Mức 3: `min` trên mọi lá.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [38]:
vals = (Solution().smallestFromLeaf(build_tree([0])), Solution().smallestFromLeaf(build_tree([1, 0])), Solution().smallestFromLeaf(build_tree([2, 1, None, 0])),
        Solution().smallestFromLeaf(build_tree([25, 0, 0])))
print(vals)
assert vals == ("a", "ab", "abc", "az")
print("Bài 1 OK")

('a', 'ab', 'abc', 'az')
Bài 1 OK


### Bài 2

In [39]:
t = build_tree([3, 1, 2, 0, None, None, 1])
ls = sorted(all_leaf_strings(t))
print(ls)
assert ls == ["abd", "bcd"] and Solution().smallestFromLeaf(t) == "abd" == oracle_tuple(tup(t))
print("Bài 2 OK: các lá cho 'abd' và 'bcd'; đáp án 'abd'")

['abd', 'bcd']
Bài 2 OK: các lá cho 'abd' và 'bcd'; đáp án 'abd'


### Bài 3

In [40]:
def dfs_answer(root):
    best = [None]

    def dfs(node, s):
        s = chr(97 + node.val) + s
        if not node.left and not node.right:
            if best[0] is None or s < best[0]:
                best[0] = s
            return
        if node.left:
            dfs(node.left, s)
        if node.right:
            dfs(node.right, s)

    dfs(root, "")
    return best[0]


assert all(dfs_answer(build_tree(l)) == w for l, w in tests3)
for n in range(1, 5):
    for t in all_trees(n, (0, 1, 25)):
        assert dfs_answer(from_tuple(t)) == oracle_tuple(t)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [41]:
class MySolutionAnswer:
    def smallestFromLeaf(self, root):
        best = None
        queue = deque([(root, chr(97 + root.val))])
        while queue:
            node, s = queue.popleft()
            if not node.left and not node.right:
                if best is None or s < best:
                    best = s
            for c in (node.left, node.right):
                if c:
                    queue.append((c, chr(97 + c.val) + s))
        return best


assert all(MySolutionAnswer().smallestFromLeaf(build_tree(l)) == w for l, w in tests3)
for n in range(1, 5):
    for t in all_trees(n, (0, 1, 25)):
        assert MySolutionAnswer().smallestFromLeaf(from_tuple(t)) == oracle_tuple(t)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [42]:
def smallest_down_answer(root):
    best, stack = None, [(root, chr(97 + root.val))]
    while stack:
        node, s = stack.pop()
        if not node.left and not node.right:
            if best is None or s < best:
                best = s
            continue
        for c in (node.left, node.right):
            if c:
                stack.append((c, s + chr(97 + c.val)))
    return best


assert smallest_down_answer(build_tree([0, 1, 2, 3, 4, 3, 4])) == "abd" and smallest_down_answer(build_tree([1, 0])) == "ba"
for n in range(1, 5):
    for t in all_trees(n, (0, 1, 25)):
        root = from_tuple(t)
        assert smallest_down_answer(root) == min(s[::-1] for s in all_leaf_strings(root))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chuỗi nhỏ nhất từ lá lên gốc → DFS lặp mang chuỗi, nối vào đầu):**

```python
class Solution:
    def smallestFromLeaf(self, root: Optional[TreeNode]) -> str:
        best = None
        stack = [(root, chr(97 + root.val))]
        while stack:
            node, s = stack.pop()
            if not node.left and not node.right:
                if best is None or s < best:
                    best = s
                continue
            for child in (node.left, node.right):
                if child:
                    stack.append((child, chr(97 + child.val) + s))
        return best
```

**Checklist trước khi submit:**

- [ ] Lá = **không có con nào** (nút một con không phải lá)?
- [ ] Nối ký tự **vào đầu** chuỗi (`ch + s`) để đọc từ lá lên gốc?
- [ ] So sánh chuỗi đầy đủ (không "nhỏ nhất của cây con + ký tự cha")?
- [ ] Biết luật tiền tố (`"abc" < "abcc"`) và Python so sánh chuỗi đúng luật đó?
- [ ] Đổi giá trị sang **chữ cái** trước khi so sánh (không so số)?
- [ ] Tránh đệ quy sâu (cây `8500` nút lệch): dùng stack hoặc tăng giới hạn?
- [ ] Đúng tên `class Solution`, `smallestFromLeaf(self, root)`, không in debug.